# Measurement DB response preprocessing

Load all competition splits in the Hugging Face `responses` and `subjects` datasets, and create dicts mapping hashes to human-readable identifiers/content.

This dataset is gated. Accept its access terms on Hugging Face and authenticate in the notebook environment first.

In [12]:
import random
import datasets

from huggingface_hub import notebook_login
notebook_login()

RANDOM_SEED = 5814
random.seed(RANDOM_SEED)

benchmarks = (
    "multi_swebench+real_webagents+"
    "researchcodebench+swe_rebench"
)

SAMPLE_SIZE = 1000

# ---------------------------------------------------------
# Create lookup tables (items + subjects)
# ---------------------------------------------------------
print("Caching items by item_id")
items_ds = datasets.load_dataset(
    "aims-foundations/measurement-db",
    name="items",
    split=benchmarks,
    token=True
)
items_lookup = {row["item_id"]: row for row in items_ds}
print(f"Items cached: {len(items_lookup)}")

print("Caching subjects by subject_id")
subjects_ds = datasets.load_dataset(
    "aims-foundations/measurement-db",
    name="subjects",
    split=benchmarks,
    token=True
)
subjects_lookup = {row["subject_id"]: row for row in subjects_ds}
print(f"Subjects cached: {len(subjects_lookup)}")


Caching items by item_id
Items cached: 8878
Caching subjects by subject_id
Subjects cached: 134


Generate one-hot vector representations for subjects

In [13]:
# dict.fromkeys preserves first-seen order while removing duplicate names.
normalized_names = list(
    dict.fromkeys(
        subject["normalized_name"]
        for subject in subjects_lookup.values()
        if subject.get("normalized_name") is not None
    )
)
num_names = len(normalized_names)
name_to_one_hot = {
    name: [int(index == position) for index in range(num_names)]
    for position, name in enumerate(normalized_names)
}

# Verify each vector's size, one-hot value, and uniqueness.
assert all(
    len(vector) == num_names and sum(vector) == 1
    for vector in name_to_one_hot.values()
)
assert len({tuple(vector) for vector in name_to_one_hot.values()}) == num_names

print(f"Created {num_names} unique name vectors; each has length {num_names}.")

Created 60 unique name vectors; each has length 60.


Sample from the `responses` dataset and merge in data from `subjects` and `items`

In [14]:

# ---------------------------------------------------------
# Streaming responses + reservoir sampling
# ---------------------------------------------------------
print("Streaming responses and performing reservoir sampling...")
responses_stream = datasets.load_dataset(
    "aims-foundations/measurement-db",
    name="responses",
    split=benchmarks,
    streaming=True,
    token=True
)

def _merge_row(example):
    item = items_lookup.get(example["item_id"])
    subject = subjects_lookup.get(example["subject_id"])
    if item is None or subject is None:
        return None
    return {**example, **item, **subject}

joined_stream = responses_stream.map(_merge_row)

reservoir = []
n_seen = 0

for row in joined_stream:
    if row is None:
        continue

    # Validate fields early
    content = row.get("content")
    subject_name = row.get("normalized_name")
    response_val = row.get("response")

    if not isinstance(content, str):
        continue
    if not isinstance(subject_name, str):
        continue
    if response_val is None:
        continue

    # Reservoir sampling
    if len(reservoir) < SAMPLE_SIZE:
        reservoir.append(row)
    else:
        j = random.randint(0, n_seen)
        if j < SAMPLE_SIZE:
            reservoir[j] = row

    n_seen += 1

print(f"Reservoir sampled {len(reservoir)} rows from ~{n_seen} streamed rows")

for item in reservoir[:5]:
    print(item)


Streaming responses and performing reservoir sampling...
Reservoir sampled 1000 rows from ~136282 streamed rows
{'subject_id': '58c3de46be1beb60', 'item_id': '0a017ffe57db9265', 'response_id': '1c80729da617b63a', 'benchmark_id': 'swe_rebench', 'trial': 3, 'test_condition': None, 'interactors': None, 'response': 0.0, 'raw_item_id': 'conda-incubator__conda-recipe-manager-49', 'content': 'Add `merge_build_host` transformation\nWe are missing a transformation of the `merge_build_host` field from CEP-14:\r\n```\r\n  # merge the build and host environments (used in many R packages on Windows)\r\n  # was `merge_build_host`\r\n  merge_build_and_host_envs: bool (defaults to false)\r\n```\r\n\r\n`rattler-build` error:\r\n```\r\nError:   × Parsing: invalid field `merge_build_host`.\r\n    ╭─[50:3]\r\n 49 │   noarch: generic\r\n 50 │   merge_build_host: $${{ true if win }}\r\n    ·   ────────┬───────\r\n    ·           ╰── here\r\n 51 │   rpaths:\r\n    ╰────\r\n```', 'asset_manifest': None, 'grad

Merge one-hot subject vectors to reservoir samples

In [15]:
# Map one-hot subject vectors back to reservoir samples
for sample in reservoir:
        sample["one_hot_normalized_name"] = name_to_one_hot[sample["normalized_name"]]

for sample in reservoir[:5]:
    print(sample["one_hot_normalized_name"])

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


Embed the item content of each sample in reservoir.

In [16]:
#from InstructorEmbedding import INSTRUCTOR
#embedder = INSTRUCTOR('hkunlp/instructor-xl')
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
instruction = "Represent the Task document for clustering: "
for sample in reservoir:
    raw = sample["content"]
    sample["content_embedding"] = embedder.encode(raw, convert_to_numpy=True)
    #= embedder.encode([[instruction, raw]], show_progress_bar=True)[0]

for sample in reservoir[:5]:
    print(sample["content_embedding"])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3809.64it/s]


[ 7.54709821e-03  4.04693522e-02  1.17786899e-02 -3.38154659e-02
  1.66037548e-02 -5.25328070e-02 -2.94179320e-02 -4.05147895e-02
 -6.09828308e-02  1.47705795e-02  4.87168320e-02 -1.26454562e-01
  1.18081607e-02  5.37168421e-02  2.83363126e-02  2.79019959e-02
 -7.90047348e-02 -1.14804832e-02  7.07449438e-03 -2.30793338e-02
 -1.69209018e-01  7.29908794e-02 -6.87941350e-03  3.45585868e-02
  7.25242272e-02 -2.06253608e-03 -7.43239671e-02  4.09175642e-02
 -2.95665395e-02 -4.94052880e-02  5.59499711e-02  8.78884196e-02
 -2.38861665e-02  2.44082715e-02  1.59421191e-01  9.62310657e-02
 -1.21163521e-02 -4.43836562e-02 -1.77326854e-02 -9.79640931e-02
  3.58260348e-02 -2.96848826e-02  2.03355011e-02 -3.95166203e-02
 -1.21467337e-01  1.57471746e-02 -1.58908535e-02 -6.99549727e-03
  2.11256053e-02 -7.46668056e-02 -1.32778455e-02  3.07533536e-02
  7.22869486e-03 -2.41631679e-02  4.48568985e-02  9.24535841e-02
 -6.18377626e-02  1.45052858e-02  2.64447392e-03 -3.49388234e-02
  2.28911508e-02  1.71839

Save finalized samples from reservoir to disk. Ready for training/testing.

In [17]:
import json
from pathlib import Path

import numpy as np

class NumpyJSONEncoder(json.JSONEncoder):
    def default(self, value):
        if isinstance(value, np.ndarray):
            return value.tolist()
        if isinstance(value, np.generic):
            return value.item()
        return super().default(value)

output_path = Path("samples.json")
with output_path.open("w", encoding="utf-8") as file:
    json.dump(reservoir, file, cls=NumpyJSONEncoder, ensure_ascii=False, indent=2)

print(f"Saved {len(reservoir)} samples to {output_path.resolve()}")

Saved 1000 samples to C:\Users\swood\Documents\VT-MEng-CSA\CS 5814 Deep Learning\Course project\PSN-IRT\samples.json


Confirm json loads into a Pandas dataframe

In [21]:
import pandas as pd
df = pd.read_json("samples.json")
print(df.head())

         subject_id           item_id       response_id       benchmark_id  \
0  58c3de46be1beb60  0a017ffe57db9265  1c80729da617b63a        swe_rebench   
1  58c3de46be1beb60  62056c796d908eb1  9666e09cc48ddce0        swe_rebench   
2  58c3de46be1beb60  2d96e651b3bb7b38  f4c5585c21ec4b10        swe_rebench   
3  58c3de46be1beb60  3a1f04e5b78c001a  7a2616011ea65ca8        swe_rebench   
4  c68b0b4cebac1009  1f858793a52e7df1  b3ce74a156734537  researchcodebench   

   trial  test_condition  interactors  response  \
0      3             NaN          NaN         0   
1     10             NaN          NaN         1   
2      7             NaN          NaN         1   
3      4             NaN          NaN         0   
4      1             NaN          NaN         1   

                                raw_item_id  \
0  conda-incubator__conda-recipe-manager-49   
1                      adbar__simplemma-114   
2  marcosschroh__dataclasses-avroschema-243   
3                   IAMconsortium__p